In [55]:
import pandas as pd
import glob
import os
import shutil

In [56]:
template_path = r"EDDtemplate\ESBasic_TRC Format.xlsx"
template_df = pd.read_excel(template_path, sheet_name="ESBasic_TRC")
template_df

,#sys_sample_code,sample_name,sample_type_code,sample_matrix_code,sample_date,sample_time,sys_loc_code,parent_sample_code,start_depth,end_depth,...,workflow_status,task_code_2,TRC_Reason_Codes,QCI,QCII,QCIII,Lab_Cert_ID_No,AltParameterCode,Approval_Code,equipment_code
0,#Text(40),Text(40),Text(20),Text(10),Date,Time,Text(20),Text(40),Numeric,Numeric,...,Text(4),Text(40),Text(50),Text(255),Text(255),Text(255),Text(255),Text(10),Text(10),Text(60)


In [57]:
dir = "data/bai_radium"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
excel_paths

['data/bai_radium\\ChemStat Import_MONBAI GWPS Radium_AK.xlsx',
 'data/bai_radium\\MONBAI_GWPS_RADIUM_EDD.xlsx']

In [58]:
dfs = [pd.read_excel(path) for path in excel_paths]
MONBAI_GWPS_RADIUM = dfs[0]
# make all of the column names lowercase for easier mapping
MONBAI_GWPS_RADIUM.columns = MONBAI_GWPS_RADIUM.columns.str.lower()
col_dict = {'report_result_text':'result_value', 'report_result_unit':"result_unit"}                     
MONBAI_GWPS_RADIUM = MONBAI_GWPS_RADIUM.rename(columns=col_dict)
MONBAI_GWPS_RADIUM['chemical_name'] = MONBAI_GWPS_RADIUM['chemical_name'].str.replace(" ", "-", regex=True)

MONBAI_GWPS_RADIUM_EDD = MONBAI_GWPS_RADIUM.reindex(columns=template_df.columns)
MONBAI_GWPS_RADIUM_EDD

,#sys_sample_code,sample_name,sample_type_code,sample_matrix_code,sample_date,sample_time,sys_loc_code,parent_sample_code,start_depth,end_depth,...,workflow_status,task_code_2,TRC_Reason_Codes,QCI,QCII,QCIII,Lab_Cert_ID_No,AltParameterCode,Approval_Code,equipment_code
0,NaN,NaN,NaN,NaN,2017-11-08 12:30:00,NaN,MW-10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,2018-01-09 09:50:00,NaN,MW-10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,2018-03-12 14:30:00,NaN,MW-10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2018-05-22 11:00:00,NaN,MW-10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,2018-05-22 11:00:00,NaN,MW-10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,NaN,NaN,NaN,NaN,2018-05-22 10:05:00,NaN,MW-9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,NaN,NaN,NaN,NaN,2018-07-25 12:30:00,NaN,MW-9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,NaN,NaN,NaN,NaN,2018-09-25 15:25:00,NaN,MW-9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,NaN,NaN,NaN,NaN,2018-11-28 13:20:00,NaN,MW-9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
# Convert sample_date to datetime to ensure standard formatting
sample_date_dt = pd.to_datetime(MONBAI_GWPS_RADIUM_EDD["sample_date"], format="%m/%d/%Y")
MONBAI_GWPS_RADIUM_EDD["sample_time"] = sample_date_dt.dt.strftime("%H:%M:%S")
# copy to sample_name column to preserve original sample names before modification
MONBAI_GWPS_RADIUM_EDD['sample_name'] = MONBAI_GWPS_RADIUM_EDD['sys_loc_code']
MONBAI_GWPS_RADIUM_EDD['result_type_code'] = "TRG"
MONBAI_GWPS_RADIUM_EDD['reportable_result'] = "Yes"
MONBAI_GWPS_RADIUM_EDD['analytic_method'] = 'TA_Ra226+228'
MONBAI_GWPS_RADIUM_EDD['cas_rn'] = "Ra226/228"
MONBAI_GWPS_RADIUM_EDD['result_unit'] = "pci/L"
MONBAI_GWPS_RADIUM_EDD["reporting_detection_limit"] = 1.0
MONBAI_GWPS_RADIUM_EDD['fraction'] = "N"
# now remove the unwanted values from sys_loc_code column
# List of values you want to remove
blank_these_locs = ["EQUIPMENT BLANK", "FIELD BLANK A", "FIELD BLANK B"]
# Filter replaces the unwanted values with an empty string
MONBAI_GWPS_RADIUM_EDD["sys_loc_code"] = MONBAI_GWPS_RADIUM_EDD["sys_loc_code"].replace(blank_these_locs, "")
MONBAI_GWPS_RADIUM_EDD['#sys_sample_code'] = MONBAI_GWPS_RADIUM_EDD["sample_name"] + "_" + sample_date_dt.dt.strftime("%Y%m%d")
MONBAI_GWPS_RADIUM_EDD['sample_type_code'] = MONBAI_GWPS_RADIUM_EDD["sample_name"].apply(lambda x: "FB" if "Field Blank" in x else "N")
MONBAI_GWPS_RADIUM_EDD['sample_type_code'] = MONBAI_GWPS_RADIUM_EDD["sample_name"].apply(lambda x: "FD" if "DUPLICATE A" in x else "N")
MONBAI_GWPS_RADIUM_EDD["sample_matrix_code"] = MONBAI_GWPS_RADIUM_EDD["sample_type_code"].apply(lambda x: "WQ" if x in ("FB", "FD") else "WG")
MONBAI_GWPS_RADIUM_EDD[MONBAI_GWPS_RADIUM_EDD.columns[MONBAI_GWPS_RADIUM_EDD.columns.str.contains("chemical_name|result", case=False)]].head(10)
# Lab SGD
# Analysis Date
# Test Type (initial, rextract, dilution, etc.)


,chemical_name,result_value,result_unit,result_type_code,reportable_result,PreservedResultValue,result_comments
0,Radium-226/228,1.42,pci/L,TRG,Yes,NaN,NaN
1,Radium-226/228,1.61,pci/L,TRG,Yes,NaN,NaN
2,Radium-226/228,1.39,pci/L,TRG,Yes,NaN,NaN
3,Radium-226/228,1.66,pci/L,TRG,Yes,NaN,NaN
4,Radium-226/228,2.41,pci/L,TRG,Yes,NaN,NaN
5,Radium-226/228,1.28,pci/L,TRG,Yes,NaN,NaN
6,Radium-226/228,2.01,pci/L,TRG,Yes,NaN,NaN
7,Radium-226/228,1.46,pci/L,TRG,Yes,NaN,NaN
8,Radium-226/228,1.60,pci/L,TRG,Yes,NaN,NaN
9,Radium-226/228,1.63,pci/L,TRG,Yes,NaN,NaN


In [60]:
# 2. Make an exact copy of the template file
output_path = r"data\bai_radium\MONBAI_GWPS_RADIUM_EDD.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    MONBAI_GWPS_RADIUM_EDD.to_excel(
        writer,
        sheet_name="ESBasic_TRC",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

In [61]:
MONBAI_GWPS_RADIUM_EDD = pd.read_excel(r"data\bai_radium\MONBAI_GWPS_RADIUM_EDD.xlsx", sheet_name="ESBasic_TRC", skiprows=[1])
MONBAI_GWPS_export = MONBAI_GWPS_RADIUM_EDD[["sys_loc_code", "cas_rn", "chemical_name", "result_value", "result_unit", "lab_qualifiers", 
                                             "reporting_detection_limit", "sample_date", "fraction"]]
MONBAI_GWPS_export = MONBAI_GWPS_export.rename(columns={
    "sys_loc_code": "Well ID", "cas_rn": "CAS Num", "chemical_name": "Analyte", "result_value": "Concentration", "result_unit": "Units", "lab_qualifiers": "Qualifier", 
    "reporting_detection_limit": "Detection Limit", "sample_date": "Sample Datetime", "fraction": "FRACTION"})
MONBAI_GWPS_export["Sample Datetime"] = MONBAI_GWPS_export["Sample Datetime"].dt.strftime("%Y/%m/%d %H:%M:%S")
MONBAI_GWPS_export["Well Type"] = "Compliance"
MONBAI_GWPS_export["Site name"] = "Monroe Power Plant"
MONBAI_GWPS_export["CCR Unit"] = MONBAI_GWPS_export["Well ID"].apply(lambda x: "Fly Ash Basin" if "16" in x else "Bottom Ash Basin")
MONBAI_GWPS_export

,Well ID,CAS Num,Analyte,Concentration,Units,Qualifier,Detection Limit,Sample Datetime,FRACTION,Well Type,Site name,CCR Unit
0,MW-10,Ra226/228,Radium-226/228,1.42,pci/L,NaN,1,2017/11/08 12:30:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
1,MW-10,Ra226/228,Radium-226/228,1.61,pci/L,NaN,1,2018/01/09 09:50:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
2,MW-10,Ra226/228,Radium-226/228,1.39,pci/L,NaN,1,2018/03/12 14:30:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
3,MW-10,Ra226/228,Radium-226/228,1.66,pci/L,NaN,1,2018/05/22 11:00:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
4,MW-10,Ra226/228,Radium-226/228,2.41,pci/L,NaN,1,2018/05/22 11:00:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
...,...,...,...,...,...,...,...,...,...,...,...,...
96,MW-9,Ra226/228,Radium-226/228,1.62,pci/L,NaN,1,2018/05/22 10:05:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
97,MW-9,Ra226/228,Radium-226/228,1.45,pci/L,NaN,1,2018/07/25 12:30:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
98,MW-9,Ra226/228,Radium-226/228,1.56,pci/L,NaN,1,2018/09/25 15:25:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin
99,MW-9,Ra226/228,Radium-226/228,1.55,pci/L,NaN,1,2018/11/28 13:20:00,N,Compliance,Monroe Power Plant,Bottom Ash Basin


In [62]:
export_path = r"C:\Users\ageglio\OneDrive - TRC\Documents\My EQuIS Work\Mdn01\CCR-Monroe Fly Ash Basin\Analytical Results II Crosstab_5.xlsx"
export = pd.read_excel(export_path)
sample_date_dt = pd.to_datetime(export["Sample Datetime"].str.strip(), format="%d %b %Y %H:%M")
export["Sample Datetime"] = sample_date_dt.dt.strftime("%Y/%m/%d %H:%M:%S")
# Remove Well IDs MW-1d, MW-3d, MW-4s, MW-5s, MW-7d, and MW-8d from the dataset
wells_to_remove = ["MW-1D", "MW-3D", "MW-4S", "MW-5S", "MW-7D", "MW-8D", "MP-001F"]
export = export[~export["Well ID"].isin(wells_to_remove)]
export['Well Type'] = "Compliance"
export["CCR Unit"] = export["LOC_GROUP_CODE"].apply(lambda x: "Bottom Ash Basin" if x == "MON_BAB_inactive" else "Fly Ash Basin" if x == "MON_FAB" else "Other")
export["Site name"] = "Monroe Power Plant"
export = pd.concat([export, MONBAI_GWPS_export], ignore_index=True)
export = export.sort_values(by=["Well ID", "Sample Datetime", "Analyte"])

# drop rows where FRACTION = "D" except for Fluoride and Sulfate analytes since those are the only two with D fractions and we want to keep them
export = export[~((export["FRACTION"] == "D") & (~export["Analyte"].isin(["Fluoride", "Sulfate"])))]
# drop LOC_TYPE and LOC_GROUP_CODE columns since they are no longer needed
export = export.drop(columns=["LOC_TYPE", "LOC_GROUP_CODE"])

In [ ]:
# 2. Make an exact copy of the template file
output_path = r"C:\Users\ageglio\OneDrive - TRC\Documents\My EQuIS Work\Mdn01\All DTE CCR Facilites\MONPP_FlatFile_AppxIV_Filt.ag.xlsx"
template_path = r"C:\Users\ageglio\OneDrive - TRC\Documents\My EQuIS Work\Mdn01\All DTE CCR Facilites\template\FlatFile_AppxIV_tempate.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    export.to_excel(
        writer,
        sheet_name="Analytical Results II Crosstab",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=True,  # <-- Set to False if your template already has the headers typed out
        startrow=0,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ageglio\\OneDrive - TRC\\Documents\\My EQuIS Work\\Mdn01\\All DTE CCR Facilites\\FlatFile_AppxIV_tempate.xlsx'

: 

In [ ]:
# Queries to check the data
# export[(export["Well ID"] == "MW-1S") & (export["Analyte"] == "Fluoride")].sort_values("Sample Datetime")